# Pipeline MNIST + API FastAPI

**Objetivo:** treinar uma MLP com TensorFlow/Keras usando `sklearn.Pipeline` + `SciKeras`, aplicar calibração (Temperature Scaling), detecção de OOD (Maximum Softmax Probability) e exportar os artefatos prontos para servir em uma API FastAPI.

**Saída deste notebook:**
- `models/mlp_mnist.keras`
- `models/preprocess_pipeline.joblib`
- `models/temperature.json`
- `models/ood_threshold.json`
- `models/metrics.csv`
- `models/confusion_matrix.png`
- Toda a estrutura `app/` da API FastAPI
- `tests/` com testes automatizados


# 1- Instala todas as dependências necessárias 

In [ ]:
%pip install tensorflow==2.15.0 --force-reinstall
%pip install -q scikeras==0.12.0 fastapi==0.104.1 "uvicorn[standard]==0.24.0" pydantic==2.5.2 pydantic-settings==2.1.0 python-multipart==0.0.6 scikit-learn==1.3.2 scipy==1.11.4 seaborn==0.13.0 matplotlib==3.8.2 joblib==1.3.2 Pillow==10.1.0 python-json-logger==2.0.7 prometheus-client==0.19.0 httpx==0.25.2 pytest==7.4.3 pytest-asyncio==0.21.1 pandas==2.1.4


  Using cached tensorflow-2.15.0-cp310-cp310-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow_intel-2.15.0-cp310-cp310-win_amd64.whl.metadata (5.1 kB)
  Using cached ml_dtypes-0.2.0-cp310-cp310-win_amd64.whl.metadata (20 kB)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached wrapt-1.14.2-cp310-cp310-win_amd64.whl.metadata (6.6 kB)
  Using cached tensorflow_io_gcs_filesystem-0.31.0-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached tensorboard-2.15.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached tensorflow_estimator-2.15.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached google_auth-2.62.0-py3-none-any.whl.metadata (6.0 kB)
  Using cached google_auth_oauthlib-1.5

# 2- Imports, seeds e diretórios do projeto

In [5]:
# ==========================================================
# Imports, seeds e diretórios do projeto
# ==========================================================
import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar
from scipy.ndimage import rotate, shift

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
from scikeras.wrappers import KerasClassifier, KerasRegressor
import joblib

# ---------- Reprodutibilidade ----------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- Diretórios ----------
# Detecta se estamos em notebooks/ ou na raiz do projeto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MODELS_DIR    = BASE_DIR / "models"
DATA_DIR      = BASE_DIR / "data"
APP_DIR       = BASE_DIR / "app"
ROUTERS_DIR   = APP_DIR / "routers"
TESTS_DIR     = BASE_DIR / "tests"
NOTEBOOKS_DIR = BASE_DIR / "notebooks"

for d in [MODELS_DIR, DATA_DIR, APP_DIR, ROUTERS_DIR, TESTS_DIR, NOTEBOOKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))
print("BASE_DIR:", BASE_DIR)

TensorFlow: 2.15.0
GPU disponível: []
BASE_DIR: c:\Users\lapri\OneDrive\Desktop\SCTEC- MEMO\PROYECTO FINAL\projeto-mnist-api
